# 44 · JV Posting Agent

**What is double-entry bookkeeping?**  
Every business transaction is recorded as two equal and opposite journal lines:
a **debit** (left-side) and a **credit** (right-side). The fundamental rule:

> Total debits must always equal total credits.

**Why does balance validation matter?**  
LLMs are good at *choosing* the right accounts (domain reasoning) but unreliable
at *arithmetic* (floating-point addition, rounding). This example separates the two:
the LLM assigns accounts and amounts; a tiny deterministic `check_balance` function
enforces the constraint. Any posting that fails the check is stamped `rejected`
before it can reach the general ledger.

In [ ]:
# Install dependencies (Colab only)
try:
    import google.colab  # noqa: F401
    %pip install -q langchain-openai langchain-core pydantic python-dotenv
except ImportError:
    pass

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"

## Part 1 — The Business Problem

Finance teams receive business events in plain English:

- *"Received IT equipment invoice from Dell, capitalise as fixed asset — $12,500"*
- *"Accrue June salary expense for finance department — $45,000"*

Converting these events into correct GL codes manually is slow,
error-prone, and scales poorly as transaction volume grows.
Common failure modes: wrong account, wrong side, imbalanced entry.

The agent automates account selection; the validator catches imbalances.

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class JournalLine(BaseModel):
    side: Literal["debit", "credit"] = Field(description="Debit or credit side")
    account_code: str = Field(description="4-digit GL account code e.g. '2100'")
    account_name: str = Field(description="Human-readable GL account name")
    amount: float = Field(description="Positive posting amount", gt=0)
    cost_centre: Optional[str] = Field(default=None, description="Cost centre e.g. CC1001")


class PostingRequest(BaseModel):
    event_description: str
    document_type: Literal["SA", "KR", "DR", "ZP", "AB", "AA", "RE"] = Field(
        description="SA=GL adj, KR=vendor invoice, DR=customer invoice, ZP=payment, AB=depreciation, AA=asset, RE=accrual"
    )
    amount: float = Field(gt=0)
    cost_centre: Optional[str] = None
    period: str = Field(description="Period YYYY-MM")


class PostingResult(BaseModel):
    lines: List[JournalLine]
    is_balanced: bool
    total_debits: float
    total_credits: float
    posting_status: Literal["approved", "rejected", "needs_review"]
    rejection_reason: Optional[str] = None


print("Schemas defined.")

## Part 2 — The Balance Calculator: Why Deterministic > LLM Arithmetic

LLMs sometimes add numbers incorrectly — especially with floating-point values.
For financial correctness we keep arithmetic out of the model entirely.
`check_balance` is a pure Python function with zero model dependency.
It uses a small `tolerance` (0.01) to absorb floating-point rounding from the model.

In [ ]:
def check_balance(debits: list, credits: list, tolerance: float = 0.01) -> bool:
    """Return True when total debits and total credits are within tolerance."""
    return abs(sum(debits) - sum(credits)) <= tolerance


# Quick sanity checks
print(check_balance([100.0], [100.0]))    # True  — balanced
print(check_balance([100.0], [99.0]))     # False — imbalanced
print(check_balance([100.0], [99.995]))   # True  — within tolerance

## Part 3 — The Workflow

The agent chain is a single `POSTING_PROMPT | llm.with_structured_output(PostingResult)`
pipeline. After the LLM responds, `run()` extracts debit and credit amounts,
calls `check_balance`, and overwrites `is_balanced` and `posting_status` with the
deterministic result — the model's own `is_balanced` field is intentionally ignored.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

CHART_OF_ACCOUNTS = {
    "1000": "Cash and Cash Equivalents",
    "1100": "Accounts Receivable",
    "1200": "Inventory",
    "1300": "Prepaid Expenses",
    "1600": "Property, Plant & Equipment",
    "1700": "Accumulated Depreciation",
    "2000": "Accounts Payable",
    "2100": "Accrued Liabilities",
    "2200": "VAT/Tax Payable",
    "2300": "Deferred Revenue",
    "3000": "Retained Earnings",
    "4000": "Revenue",
    "4100": "Service Revenue",
    "5000": "Cost of Goods Sold",
    "6000": "Salaries & Wages",
    "6100": "Rent Expense",
    "6200": "Depreciation Expense",
    "6300": "Interest Expense",
    "6400": "G&A Expenses",
    "7000": "Marketing Expense",
}

POSTING_PROMPT = SystemMessage(
    "You are a senior GAAP accountant. Prepare the double-entry journal posting "
    "for the given business event.\n"
    "Rules:\n"
    "- Every posting MUST have at least one debit and one credit line\n"
    "- Total debits MUST equal total credits\n"
    "- Use the most specific account available\n"
    "- Include cost_centre on all P&L accounts (4xxx-8xxx) when provided\n"
    "Chart of accounts:\n"
    + "\n".join(f"  {k}: {v}" for k, v in CHART_OF_ACCOUNTS.items())
)


def create_posting_agent():
    llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
    return POSTING_PROMPT | llm.with_structured_output(PostingResult)


def run(request: PostingRequest) -> PostingResult:
    agent = create_posting_agent()
    result = agent.invoke(
        f"Event: {request.event_description}\n"
        f"Document type: {request.document_type}\n"
        f"Amount: ${request.amount:,.2f}\n"
        f"Period: {request.period}\n"
        f"Cost centre: {request.cost_centre or 'N/A'}"
    )
    debits = [line.amount for line in result.lines if line.side == "debit"]
    credits = [line.amount for line in result.lines if line.side == "credit"]
    balanced = check_balance(debits, credits)
    return PostingResult(
        lines=result.lines,
        is_balanced=balanced,
        total_debits=round(sum(debits), 2),
        total_credits=round(sum(credits), 2),
        posting_status="approved" if balanced else "rejected",
        rejection_reason=(
            None
            if balanced
            else f"Imbalance: debits ${sum(debits):,.2f} vs credits ${sum(credits):,.2f}"
        ),
    )


print("Workflow ready.")

## Part 4 — Run Two Samples

In [ ]:
# Sample 1: Asset capitalisation
req1 = PostingRequest(
    event_description="Received IT equipment invoice from Dell, capitalise as fixed asset",
    document_type="AA",
    amount=12500.00,
    cost_centre="CC5001",
    period="2025-06",
)
result1 = run(req1)
print(f"Status  : {result1.posting_status.upper()}")
print(f"Balanced: {result1.is_balanced}")
for line in result1.lines:
    print(f"  {line.side:<6} {line.account_code} {line.account_name:<35} {line.amount:>10,.2f}  {line.cost_centre or ''}")
print(f"DR {result1.total_debits:,.2f}  CR {result1.total_credits:,.2f}")

In [ ]:
# Sample 2: Salary accrual
req2 = PostingRequest(
    event_description="Accrue June salary expense for finance department, not yet paid",
    document_type="RE",
    amount=45000.00,
    cost_centre="CC2001",
    period="2025-06",
)
result2 = run(req2)
print(f"Status  : {result2.posting_status.upper()}")
print(f"Balanced: {result2.is_balanced}")
for line in result2.lines:
    print(f"  {line.side:<6} {line.account_code} {line.account_name:<35} {line.amount:>10,.2f}  {line.cost_centre or ''}")
print(f"DR {result2.total_debits:,.2f}  CR {result2.total_credits:,.2f}")

## Exercise: Intentional Imbalance

What happens when the debit and credit totals don't match?
Try calling `check_balance` directly with mismatched amounts.
What would `posting_status` be in that case?

**Your turn:** call `check_balance([5000], [4999])` — what do you get?

In [ ]:
# Answer key
answer = check_balance([5000], [4999])
print(f"check_balance([5000], [4999]) => {answer}")  # False
status = "approved" if answer else "rejected"
print(f"posting_status would be: {status}")  # rejected